In [1]:
import numpy as np
import pandas as pd
import torch
import os

import seaborn as sns
import matplotlib.pyplot as plt
import ot
import re

In [2]:
def rename_cell_type(cell_type):
    return re.sub(r'[^a-zA-Z0-9\/+&]', '_', cell_type)

def aggregate_attention_scores(attention_scores, cell_type_matrix):
    """Compute the sum of attention scores for each sender-receiver cell type pair 
    and normalize by the number of receiver cells of each type, ensuring float values."""
    
    # Extract receiver and sender cell types
    receiver_types = cell_type_matrix[:, 0]  # First column: receiver cell type
    sender_types = cell_type_matrix[:, 1:]  # Other columns: sender cell type

    # Flatten data to count interactions
    data = []
    for i in range(attention_scores.shape[0]):
        for j in range(attention_scores.shape[1]):
            data.append((receiver_types[i], sender_types[i, j], float(attention_scores[i, j])))

    df = pd.DataFrame(data, columns=["Receiver", "Sender", "Score"])

    # Compute sum of attention scores for each sender-receiver pair
    sum_scores = df.groupby(["Sender", "Receiver"])["Score"].sum().unstack(fill_value=0)

    # Count the number of receiver cells per receiver type
    receiver_counts = df.groupby("Receiver").size()

    # Normalize by the number of receiver cells per receiver type
    normalized_df = sum_scores.div(receiver_counts, axis=1).fillna(0)

    # Rename cell types
    normalized_df.index = [rename_cell_type(idx) for idx in normalized_df.index]
    normalized_df.columns = [rename_cell_type(col) for col in normalized_df.columns]

    # Aggregate after renaming
    normalized_df = normalized_df.groupby(normalized_df.index).sum()
    normalized_df = normalized_df.groupby(normalized_df.columns, axis=1).sum()
    
    # Sort the dataframe
    sorted_index = sorted(normalized_df.index)
    sorted_columns = sorted(normalized_df.columns)
    normalized_df = normalized_df.loc[sorted_index, sorted_columns]

    # Convert all values to float explicitly
    normalized_df = normalized_df.astype(float)

    return normalized_df

# mouse

In [3]:
def drop_unlabeled(df):
    """
    Remove rows and columns labeled as 'Unlabeled' from a square pandas DataFrame.
    """
    return df.loc[~df.index.isin(['other']), ~df.columns.isin(['other'])]

In [4]:
# GITIII
results=torch.load("/gpfs/gibbs/pi/zhao/xx244/GITIII_backup/Mouse_brain_evaluate/edges/"+"edges_"+"mouse1_slice201"+".pth",map_location=torch.device('cpu'))

cell_type_matrix=np.array(results['cell_type_name'])
print(np.unique(cell_type_matrix[:,0]))

attention_scores=results["attention_score"]/8
attention_scores=torch.abs(attention_scores)
attention_scores=attention_scores/torch.sum(attention_scores,dim=(0,1),keepdim=True)
attention_scores=torch.mean(attention_scores,dim=-1)
#attention_scores=attention_scores/torch.sum(attention_scores,dim=-1,keepdim=True)
print(attention_scores.shape,attention_scores)

GITIII=aggregate_attention_scores(attention_scores, cell_type_matrix)
GITIII=drop_unlabeled(GITIII)
print(GITIII)

GITIII.to_csv("./overall_strength/mouse.csv")

['Astro' 'Endo' 'L2/3 IT' 'L4/5 IT' 'L5 ET' 'L5 IT' 'L5/6 NP' 'L6 CT'
 'L6 IT' 'L6 IT Car3' 'L6b' 'Lamp5' 'Micro' 'OPC' 'Oligo' 'PVM' 'Peri'
 'Pvalb' 'SMC' 'Sncg' 'Sst' 'VLMC' 'Vip' 'other']
torch.Size([6137, 49]) tensor([[5.6957e-06, 5.0846e-05, 2.9702e-05,  ..., 2.7433e-06, 1.2594e-06,
         2.3229e-06],
        [1.9175e-05, 4.5652e-06, 1.0887e-05,  ..., 3.0700e-06, 7.9239e-07,
         7.9229e-07],
        [1.2219e-05, 4.8622e-06, 4.3247e-06,  ..., 1.8577e-05, 8.8528e-07,
         8.8672e-07],
        ...,
        [2.9272e-05, 1.1446e-05, 2.9795e-06,  ..., 1.7537e-06, 1.7662e-06,
         2.2040e-06],
        [4.4383e-06, 1.6576e-06, 1.5294e-06,  ..., 1.4758e-06, 1.5326e-05,
         1.1482e-06],
        [5.0083e-06, 7.6753e-06, 5.7411e-06,  ..., 1.4445e-06, 1.1572e-06,
         7.1328e-06]])
                   Astro          Endo       L2/3_IT       L4/5_IT  \
Astro       2.518719e-07  2.035605e-07  5.107847e-07  2.589977e-07   
Endo        1.899658e-07  2.666232e-07  2.722005e-

/tmp/ipykernel_3724478/3096831869.py:35: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  normalized_df = normalized_df.groupby(normalized_df.columns, axis=1).sum()


# AD

In [5]:
# GITIII
results=torch.load("/gpfs/gibbs/pi/zhao/xx244/GITIII_backup/AD_evaluate/edges/"+"edges_"+"H20.33.001.CX28.MTG.02.007.1.02.03"+".pth",map_location=torch.device('cpu'))

cell_type_matrix=np.array(results['cell_type_name'])
print(np.unique(cell_type_matrix[:,0]))

attention_scores=results["attention_score"]/8
attention_scores=torch.abs(attention_scores)
attention_scores=attention_scores/torch.sum(attention_scores,dim=(0,1),keepdim=True)
attention_scores=torch.mean(attention_scores,dim=-1)
#attention_scores=attention_scores/torch.sum(attention_scores,dim=-1,keepdim=True)
print(attention_scores.shape,attention_scores)

GITIII=aggregate_attention_scores(attention_scores, cell_type_matrix)
GITIII=drop_unlabeled(GITIII)
print(GITIII)

GITIII.to_csv("./overall_strength/AD.csv")

['Astrocyte' 'Chandelier' 'Endothelial' 'L2/3 IT' 'L4 IT' 'L5 ET' 'L5 IT'
 'L5/6 NP' 'L6 CT' 'L6 IT' 'L6 IT Car3' 'L6b' 'Lamp5' 'Lamp5 Lhx6'
 'Microglia-PVM' 'OPC' 'Oligodendrocyte' 'Pax6' 'Pvalb' 'Sncg' 'Sst'
 'Sst Chodl' 'VLMC' 'Vip']
torch.Size([15222, 49]) tensor([[7.9219e-06, 1.3390e-06, 3.6029e-06,  ..., 5.2694e-07, 3.0335e-06,
         1.7086e-06],
        [9.2649e-06, 5.8672e-06, 1.0478e-06,  ..., 1.6327e-06, 2.7177e-07,
         1.0177e-06],
        [7.5088e-06, 8.4534e-06, 3.3843e-06,  ..., 1.9266e-06, 7.4721e-07,
         5.1320e-07],
        ...,
        [1.9784e-06, 2.5197e-06, 3.7239e-06,  ..., 9.0013e-07, 8.1124e-06,
         8.6420e-07],
        [1.9580e-06, 5.6676e-07, 9.9174e-07,  ..., 5.4137e-07, 3.8416e-07,
         4.1162e-07],
        [1.8344e-05, 4.5189e-06, 2.5698e-06,  ..., 3.0061e-06, 6.0843e-07,
         2.8442e-06]])
                    Astrocyte    Chandelier   Endothelial       L2/3_IT  \
Astrocyte        1.222763e-07  1.124405e-07  8.913709e-08  2.434996e

/tmp/ipykernel_3724478/3096831869.py:35: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  normalized_df = normalized_df.groupby(normalized_df.columns, axis=1).sum()


# NSCLC

In [6]:
# GITIII
results=torch.load("/gpfs/gibbs/pi/zhao/xx244/GITIII_backup/NSCLC_evaluate/edges/"+"edges_"+"Lung6"+".pth",map_location=torch.device('cpu'))

cell_type_matrix=np.array(results['cell_type_name'])
print(np.unique(cell_type_matrix[:,0]))

attention_scores=results["attention_score"]/8
attention_scores=torch.abs(attention_scores)
attention_scores=attention_scores/torch.sum(attention_scores,dim=(0,1),keepdim=True)
attention_scores=torch.mean(attention_scores,dim=-1)
#attention_scores=attention_scores/torch.sum(attention_scores,dim=-1,keepdim=True)
print(attention_scores.shape,attention_scores)

GITIII=aggregate_attention_scores(attention_scores, cell_type_matrix)
GITIII=drop_unlabeled(GITIII)
print(GITIII)

GITIII.to_csv("./overall_strength/NSCLC.csv")

['B-cell' 'NK' 'T CD4 memory' 'T CD4 naive' 'T CD8 memory' 'T CD8 naive'
 'Treg' 'endothelial' 'epithelial' 'fibroblast' 'mDC' 'macrophage' 'mast'
 'monocyte' 'neutrophil' 'pDC' 'plasmablast' 'tumor 12' 'tumor 13'
 'tumor 5' 'tumor 6' 'tumor 9']
torch.Size([89091, 49]) tensor([[1.6198e-06, 1.2169e-06, 4.7957e-07,  ..., 1.0265e-07, 5.8994e-08,
         7.8496e-08],
        [2.6525e-06, 6.3357e-07, 2.5617e-07,  ..., 1.3824e-07, 5.3094e-08,
         9.4552e-08],
        [3.0320e-06, 1.5755e-06, 5.6262e-07,  ..., 7.3587e-08, 5.2325e-08,
         5.6888e-08],
        ...,
        [1.8167e-06, 5.8376e-07, 3.1634e-07,  ..., 3.1839e-07, 6.6032e-08,
         2.6326e-07],
        [2.4924e-06, 2.5105e-06, 1.1893e-06,  ..., 6.1160e-08, 6.5681e-08,
         1.2626e-07],
        [2.3296e-06, 1.0330e-06, 5.3198e-07,  ..., 1.2070e-07, 2.3255e-07,
         5.7919e-08]])


/tmp/ipykernel_3724478/3096831869.py:35: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  normalized_df = normalized_df.groupby(normalized_df.columns, axis=1).sum()


                    B_cell            NK  T_CD4_memory   T_CD4_naive  \
B_cell        2.155459e-08  1.877879e-09  4.359063e-09  9.537099e-09   
NK            1.037415e-09  3.008666e-09  1.183789e-09  1.239940e-09   
T_CD4_memory  5.834072e-09  4.685325e-09  6.783830e-09  7.066637e-09   
T_CD4_naive   1.139969e-08  2.892156e-09  5.862245e-09  1.221130e-08   
T_CD8_memory  7.045568e-09  4.677776e-09  3.737795e-09  4.733895e-09   
T_CD8_naive   1.292756e-08  7.097782e-09  8.999622e-09  1.194721e-08   
Treg          6.736159e-09  3.198155e-09  3.182620e-09  5.581991e-09   
endothelial   1.351191e-08  1.210004e-08  1.730007e-08  1.317770e-08   
epithelial    2.416624e-08  1.792715e-08  2.048142e-08  1.881746e-08   
fibroblast    1.154593e-08  1.393538e-08  2.832096e-08  1.156064e-08   
mDC           6.807798e-09  4.367970e-09  6.203267e-09  6.913800e-09   
macrophage    1.852088e-08  2.666839e-08  1.694544e-08  2.017747e-08   
mast          1.449909e-09  8.958781e-10  1.425956e-09  1.553524

# BC

In [7]:
def drop_unlabeled(df):
    """
    Remove rows and columns labeled as 'Unlabeled' from a square pandas DataFrame.
    """
    return df.loc[~df.index.isin(["Unlabeled"]), ~df.columns.isin(["Unlabeled"])]

In [8]:
# GITIII
results=torch.load("/gpfs/gibbs/pi/zhao/xx244/GITIII_backup/BC_evaluate/edges/"+"edges_"+"sample1_rep1"+".pth",map_location=torch.device('cpu'))

attention_scores=results["attention_score"]/8
attention_scores=torch.abs(attention_scores)
attention_scores=attention_scores/torch.sum(attention_scores,dim=(0,1),keepdim=True)
attention_scores=torch.mean(attention_scores,dim=-1)
#attention_scores=attention_scores/torch.sum(attention_scores,dim=-1,keepdim=True)
print(attention_scores.shape,attention_scores)

cell_type_matrix=np.array(results['cell_type_name'])

GITIII=aggregate_attention_scores(attention_scores, cell_type_matrix)
print(GITIII)

GITIII.to_csv("./overall_strength/BC.csv")

torch.Size([159224, 49]) tensor([[1.5545e-06, 5.9569e-07, 4.6330e-07,  ..., 1.4886e-08, 2.2926e-08,
         1.4915e-08],
        [1.0743e-06, 1.8927e-07, 1.0787e-07,  ..., 1.8086e-08, 5.7069e-08,
         1.7890e-08],
        [8.1149e-07, 7.3008e-07, 9.3261e-07,  ..., 3.9842e-08, 2.2961e-08,
         3.5115e-08],
        ...,
        [5.2520e-07, 4.7792e-07, 1.6218e-07,  ..., 2.5713e-08, 2.5725e-08,
         2.5726e-08],
        [7.8703e-07, 6.7627e-07, 2.3634e-07,  ..., 4.6419e-08, 3.2024e-08,
         3.0796e-08],
        [4.2227e-07, 5.2029e-07, 2.7060e-07,  ..., 2.5415e-08, 2.5418e-08,
         2.5424e-08]])


/tmp/ipykernel_3724478/3096831869.py:35: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  normalized_df = normalized_df.groupby(normalized_df.columns, axis=1).sum()


                              B_Cells  CD4+_T_Cells  CD8+_T_Cells  \
B_Cells                  1.325462e-08  7.934296e-09  6.977384e-09   
CD4+_T_Cells             1.964871e-08  1.950693e-08  1.733232e-08   
CD8+_T_Cells             1.487224e-08  1.337560e-08  1.193300e-08   
DCIS_1                   3.744225e-10  3.510883e-10  1.236272e-09   
DCIS_2                   5.330375e-10  1.783514e-10  1.683177e-09   
Endothelial              7.074042e-09  6.028614e-09  6.137446e-09   
IRF7+_DCs                1.593067e-09  1.154812e-09  7.948057e-10   
Invasive_Tumor           3.048099e-09  1.293380e-09  4.230277e-09   
LAMP3+_DCs               7.213034e-10  1.406516e-09  7.882954e-10   
Macrophages_1            1.252065e-08  9.523417e-09  1.158596e-08   
Macrophages_2            3.742668e-09  1.135274e-09  1.520308e-09   
Mast_Cells               3.047736e-10  9.707616e-11  1.493289e-10   
Myoepi_ACTA2+            1.439694e-09  6.224485e-10  2.526742e-09   
Myoepi_KRT15+            1.370002e